## operator decomposition
 This notebook is used to derive operator decomposition in spin orbiral format

## Defination of normal order operator
Eq(35)

In [119]:
from sympy.core.expr import Expr
from sympy import sympify,symbols
from sympy.core.containers import Tuple
from sympy import latex
from sympy.printing.latex import LatexPrinter, print_latex
from sympy.combinatorics import Permutation,Cycle
from sympy import Add ,Mul


In [120]:
class TensorSymbol(Expr):
    is_commutative=True
class Tensor(TensorSymbol):
    def __new__(cls,symbol,upper,lower):
        symbo=sympify(symbol)
        upper=Tuple(*upper)
        lower=Tuple(*lower)
        return TensorSymbol.__new__(cls,symbol,upper,lower)
    def _latex(self,printer):
        return "%s^{%s}_{%s}" %(
            self.symbol,
            "".join([ i.name for  i in  self.args[1]]),
            "".join([ i.name for  i in  self.args[2]]),
    
        )
    @property
    def symbol(self):
        return self.args[0]
    @property
    def upper(self):
        return self.args[1]
    @property
    def lower(self):
        return self.args[2]
    def __repr__(self):
        return "%s(%s,%s)" % self.args
    def _print(self):
        return "%s(%s,%s)" % self.args

In [121]:
p,q,r,s,t,u=symbols("p q r s t u")
gamma=symbols("\\gamma")
op=symbols("a")
nop=symbols("\\tilde{a}")
lambd=symbols("\\lambda")

In [122]:

dm1=Tensor(gamma,(p,),(q,))
dm2=Tensor(gamma,(p,r),(q,s))
dm3=Tensor(gamma,(p,r,t),(q,s,u))

op1=Tensor(op,(p,),(q,))
op2=Tensor(op,(p,r),(q,s))


nop1=Tensor(nop,(p,),(q,))
nop2=Tensor(nop,(p,r),(q,s))



In [123]:
def gen_perm(i):
    total_ind=[0,1,2]
    total_ind.pop(i)
    perm=Permutation([*(total_ind),i])
    return perm.list(),perm.signature()


In [124]:
upper=[p,q,r]
lower=[s,t,u]

In [125]:
exprs1=[]
for j in range(3):
    for i in range(3):
        upper_ind,upper_sig=gen_perm(i)
        lower_ind,lower_sig=gen_perm(j)
        sign=upper_sig*lower_sig
        ## fix position
        if i==2 and j==0:
            # swap ind
            assert upper_ind==[0,1,2]
            upper_ind=[1,0,2]
            sign*=-1
        if j==2 and i==0:
            # swap ind
            assert lower_ind==[0,1,2]
            lower_ind=[1,0,2]
            sign*=-1
        expr=sign*Tensor(gamma,(upper[upper_ind[-1]],),(lower[lower_ind[-1]],))*Tensor(
            nop,(upper[upper_ind[0]],upper[upper_ind[1]]),(lower[lower_ind[0]],lower[lower_ind[1]])
        )
        exprs1.append(expr)

In [126]:
# with open("normal_order.tex","w") as f:
#     for expr in exprs:
#         f.write(latex(expr))
#         f.write("\\\\\n")

In [127]:
from sympy.combinatorics.subsets import ksubsets

In [128]:
total_ind=[0,1,2]
def gen_perm(subset):
    others=[i for i in total_ind if i not in subset]
    perm=[]
    perm.extend(others)
    perm.extend(subset)
    perm=Permutation(perm)
    return perm.list(),perm.signature()
exprs2=[]
for i in ksubsets(total_ind,2):
    for j in ksubsets(total_ind,2):
        upper_ind,upper_sig=gen_perm(i)
        lower_ind,lower_sig=gen_perm(j) 
        sign=upper_sig*lower_sig 
        ## fix position
        swap=Permutation([0,2,1])
        if i==(0,1) and j==(1,2):
            lower_ind=swap(lower_ind)
            sign*=swap.signature()
        if i==(1,2) and j==(0,1):
            upper_ind=swap(upper_ind)
            sign*=swap.signature()
        expr=sign*Tensor(gamma,(upper[upper_ind[-2]],upper[upper_ind[-1]]),
                         (lower[lower_ind[-2]],lower[lower_ind[-1]]))*Tensor(
            nop,(upper[upper_ind[0]],),(lower[lower_ind[0]],))
        exprs2.append(expr)


In [129]:
import re

def split_terms(expression):
    """
    Splits a LaTeX expression at top-level '+' or '-' signs,
    preserving the sign with each term.
    """
    term_begin=None
    term_end=None
    terms=[]
    for ind,ch in enumerate(expression):

        if (ch == '+' or ch == '-') :
            if term_begin is None:
                term_begin=ind ##initial terms
                if term_begin==0:
                    continue
                else:
                    term_begin=0
                    term_end=ind
                    new_term=expression[term_begin:term_end]
                    terms.append(new_term)
                    term_begin=term_end
            else:
                term_end=ind
                new_term=expression[term_begin:term_end]
                terms.append(new_term)
                term_begin=term_end
        if ind==len(expression)-1:
            ## last term 
            last_term=expression[term_begin:]
            terms.append(last_term)
    return terms

In [130]:
op3=Tensor(op,upper,lower)
nop3=Tensor(nop,upper,lower)
dm3=Tensor(gamma,upper,lower)
final_expr3=Add(nop3,*exprs1,*exprs2,dm3)
terms=split_terms(latex(final_expr3))
with open("normal_order3.tex","w") as f:
        f.write(latex(op3)+"=")
        for ind ,term in enumerate(terms):
            f.write(term)
            if ind%5==4:
                  f.write("\\\\\n")

## two body operator

In [131]:
upper=[p,q,]
lower=[s,t,]

In [132]:
def gen_perm(i):
    total_ind=[0,1]
    total_ind.pop(i)
    perm=Permutation([*(total_ind),i])
    return perm.list(),perm.signature()

In [133]:
exprs1=[]
for j in range(2):
    for i in range(2):
        upper_ind,upper_sig=gen_perm(i)
        lower_ind,lower_sig=gen_perm(j)
        sign=upper_sig*lower_sig
        expr=sign*Tensor(gamma,(upper[upper_ind[-1]],),(lower[lower_ind[-1]],))*Tensor(
            nop,(upper[upper_ind[0]],),(lower[lower_ind[0]],)
        )
        exprs1.append(expr)

In [134]:
op2=Tensor(op,upper,lower)
nop2=Tensor(nop,upper,lower)
dm2=Tensor(gamma,upper,lower)

In [135]:
final_expr2=Add(nop2,*exprs1,dm2)
with open("normal_order2.tex","w") as f:
        f.write(latex(op2)+"="+latex(final_expr2))

## one body opeartor

In [136]:
upper=[p,]
lower=[s,]
op1=Tensor(op,upper,lower)
nop1=Tensor(nop,upper,lower)
dm1=Tensor(gamma,upper,lower)

In [137]:
final_expr1=Add(nop1,dm1)

with open("normal_order1.tex","w") as f:
        f.write(latex(op1)+"="+latex(final_expr1))

## change normal operator to operator

In [138]:
sub_expr1=Add(op1,-dm1)

In [139]:
with open("substitution1.tex","w") as f:
    f.write(latex(nop1)+"="+
            latex(sub_expr1))

In [140]:
sub_expr2=Add(op2,-dm2,*[-i for i in exprs1])
with open("substitution2.tex","w") as f:
    f.write(latex(nop2)+"="+
            latex(sub_expr2))

In [141]:
p1,q1,r1,s1,t1,u1=symbols("p1 q1 r1 s1 t1 u1")

In [142]:
sub_expr1

-\gamma((p,),(s,)) + a((p,),(s,))

In [143]:
sub_expr2

-\gamma((p,),(s,))*\tilde{a}((q,),(t,)) + \gamma((p,),(t,))*\tilde{a}((q,),(s,)) + \gamma((q,),(s,))*\tilde{a}((p,),(t,)) - \gamma((q,),(t,))*\tilde{a}((p,),(s,)) - \gamma((p, q),(s, t)) + a((p, q),(s, t))

In [144]:
sub_expr2=sub_expr2.subs(p,p1)
sub_expr2=sub_expr2.subs(q,q1)
sub_expr2=sub_expr2.subs(s,s1)
sub_expr2=sub_expr2.subs(t,t1)

In [145]:
sub_expr2

-\gamma((p1,),(s1,))*\tilde{a}((q1,),(t1,)) + \gamma((p1,),(t1,))*\tilde{a}((q1,),(s1,)) + \gamma((q1,),(s1,))*\tilde{a}((p1,),(t1,)) - \gamma((q1,),(t1,))*\tilde{a}((p1,),(s1,)) - \gamma((p1, q1),(s1, t1)) + a((p1, q1),(s1, t1))

In [146]:
sub_expr1=sub_expr1.subs(p,p1)
sub_expr1=sub_expr1.subs(s,s1)

In [147]:
sub_expr1

-\gamma((p1,),(s1,)) + a((p1,),(s1,))

In [148]:
final_expr3

\gamma((p,),(s,))*\tilde{a}((q, r),(t, u)) - \gamma((p,),(t,))*\tilde{a}((q, r),(s, u)) - \gamma((p,),(u,))*\tilde{a}((q, r),(t, s)) - \gamma((q,),(s,))*\tilde{a}((p, r),(t, u)) + \gamma((q,),(t,))*\tilde{a}((p, r),(s, u)) - \gamma((q,),(u,))*\tilde{a}((p, r),(s, t)) - \gamma((r,),(s,))*\tilde{a}((q, p),(t, u)) - \gamma((r,),(t,))*\tilde{a}((p, q),(s, u)) + \gamma((r,),(u,))*\tilde{a}((p, q),(s, t)) + \gamma((p, q),(s, t))*\tilde{a}((r,),(u,)) - \gamma((p, q),(s, u))*\tilde{a}((r,),(t,)) - \gamma((p, q),(u, t))*\tilde{a}((r,),(s,)) - \gamma((p, r),(s, t))*\tilde{a}((q,),(u,)) + \gamma((p, r),(s, u))*\tilde{a}((q,),(t,)) - \gamma((p, r),(t, u))*\tilde{a}((q,),(s,)) - \gamma((q, r),(s, u))*\tilde{a}((p,),(t,)) + \gamma((q, r),(t, u))*\tilde{a}((p,),(s,)) - \gamma((r, q),(s, t))*\tilde{a}((p,),(u,)) + \gamma((p, q, r),(s, t, u)) + \tilde{a}((p, q, r),(s, t, u))

In [149]:
def substitute_term(test_term):
    test_factor=[]
    nop2_inds=[]
    for ind,factor in enumerate(test_term.args):
        test_factor.append(factor)
        if isinstance(factor,Tensor) and factor.symbol==nop and len(factor.upper)==2 :
            nop2_inds.append(ind)
    for nop2_ind in nop2_inds:
        new_factor=sub_expr2.subs(p1,test_factor[nop2_ind].upper[0])
        new_factor=new_factor.subs(q1,test_factor[nop2_ind].upper[1])
        new_factor=new_factor.subs(s1,test_factor[nop2_ind].lower[0])
        new_factor=new_factor.subs(t1,test_factor[nop2_ind].lower[1])
        test_factor[nop2_ind]=new_factor
    new_test_term=Mul(*test_factor)
    return new_test_term.expand()

In [150]:
expr_sub2_only=[]
for arg in final_expr3.args:
    ## loop over summation 
    if isinstance(arg,Mul):
        new_arg= substitute_term(arg)
    else:
        new_arg=arg
    expr_sub2_only.append(new_arg)

In [151]:
expr_sub2_only=Add(*expr_sub2_only)

In [152]:
expr_sub2_only

-2*\gamma((p,),(s,))*\gamma((q,),(t,))*\tilde{a}((r,),(u,)) + 2*\gamma((p,),(s,))*\gamma((q,),(u,))*\tilde{a}((r,),(t,)) + 2*\gamma((p,),(s,))*\gamma((r,),(t,))*\tilde{a}((q,),(u,)) - 2*\gamma((p,),(s,))*\gamma((r,),(u,))*\tilde{a}((q,),(t,)) - \gamma((p,),(s,))*\gamma((q, r),(t, u)) + \gamma((p,),(s,))*a((q, r),(t, u)) + 2*\gamma((p,),(t,))*\gamma((q,),(s,))*\tilde{a}((r,),(u,)) - 2*\gamma((p,),(t,))*\gamma((q,),(u,))*\tilde{a}((r,),(s,)) - 2*\gamma((p,),(t,))*\gamma((r,),(s,))*\tilde{a}((q,),(u,)) + 2*\gamma((p,),(t,))*\gamma((r,),(u,))*\tilde{a}((q,),(s,)) + \gamma((p,),(t,))*\gamma((q, r),(s, u)) - \gamma((p,),(t,))*a((q, r),(s, u)) - 2*\gamma((p,),(u,))*\gamma((q,),(s,))*\tilde{a}((r,),(t,)) + 2*\gamma((p,),(u,))*\gamma((q,),(t,))*\tilde{a}((r,),(s,)) + 2*\gamma((p,),(u,))*\gamma((r,),(s,))*\tilde{a}((q,),(t,)) - 2*\gamma((p,),(u,))*\gamma((r,),(t,))*\tilde{a}((q,),(s,)) + \gamma((p,),(u,))*\gamma((q, r),(t, s)) - \gamma((p,),(u,))*a((q, r),(t, s)) - 2*\gamma((q,),(s,))*\gamma((r,

In [153]:
def substitute_term1(test_term):
    test_factor=[]
    nop1_inds=[]
    for ind,factor in enumerate(test_term.args):
        test_factor.append(factor)
        if isinstance(factor,Tensor) and factor.symbol==nop and len(factor.upper)==1 :
            nop1_inds.append(ind)
    for nop1_ind in nop1_inds:
        new_factor=sub_expr1.subs(p1,test_factor[nop1_ind].upper[0])
        new_factor=new_factor.subs(s1,test_factor[nop1_ind].lower[0])
        test_factor[nop1_ind]=new_factor
    new_test_term=Mul(*test_factor)
    return new_test_term.expand()

In [154]:
test_term=expr_sub2_only.args[10]

In [155]:
test_term

\gamma((p, r),(s, u))*\tilde{a}((q,),(t,))

In [156]:
substitute_term1(test_term)

-\gamma((q,),(t,))*\gamma((p, r),(s, u)) + \gamma((p, r),(s, u))*a((q,),(t,))

In [157]:
expr_sub1_only=[]
for arg in expr_sub2_only.args:
    if isinstance(arg,Mul):
        new_arg= substitute_term1(arg)
    else:
        new_arg=arg
    expr_sub1_only.append(new_arg)

In [158]:
expr_sub_final=Add(*expr_sub1_only)

In [159]:
expr_sub_final

6*\gamma((p,),(s,))*\gamma((q,),(t,))*\gamma((r,),(u,)) - 2*\gamma((p,),(s,))*\gamma((q,),(t,))*a((r,),(u,)) - 6*\gamma((p,),(s,))*\gamma((q,),(u,))*\gamma((r,),(t,)) + 2*\gamma((p,),(s,))*\gamma((q,),(u,))*a((r,),(t,)) + 2*\gamma((p,),(s,))*\gamma((r,),(t,))*a((q,),(u,)) - 2*\gamma((p,),(s,))*\gamma((r,),(u,))*a((q,),(t,)) - 2*\gamma((p,),(s,))*\gamma((q, r),(t, u)) + \gamma((p,),(s,))*a((q, r),(t, u)) - 6*\gamma((p,),(t,))*\gamma((q,),(s,))*\gamma((r,),(u,)) + 2*\gamma((p,),(t,))*\gamma((q,),(s,))*a((r,),(u,)) + 6*\gamma((p,),(t,))*\gamma((q,),(u,))*\gamma((r,),(s,)) - 2*\gamma((p,),(t,))*\gamma((q,),(u,))*a((r,),(s,)) - 2*\gamma((p,),(t,))*\gamma((r,),(s,))*a((q,),(u,)) + 2*\gamma((p,),(t,))*\gamma((r,),(u,))*a((q,),(s,)) + 2*\gamma((p,),(t,))*\gamma((q, r),(s, u)) - \gamma((p,),(t,))*a((q, r),(s, u)) + 6*\gamma((p,),(u,))*\gamma((q,),(s,))*\gamma((r,),(t,)) - 2*\gamma((p,),(u,))*\gamma((q,),(s,))*a((r,),(t,)) - 6*\gamma((p,),(u,))*\gamma((q,),(t,))*\gamma((r,),(s,)) + 2*\gamma((p,)

In [160]:
terms=split_terms(latex(expr_sub_final))

In [161]:
with open("substitution3.tex","w") as f:
    f.write(latex(op3)+"=")
    for ind ,term in enumerate(terms):
        f.write(term)
        if ind%5==4:
                f.write("\\\\\n") 

## Cumulant relationship

In [162]:
cum3=Tensor(lambd,[p,q,r],[s,t,u])

In [163]:
cum3

\lambda((p, q, r),(s, t, u))

In [164]:
upper=[p,q,r]
lower=[s,t,u]

In [165]:
def gen_perm(i):
    total_ind=[0,1,2]
    total_ind.pop(i)
    perm=Permutation([*(total_ind),i])
    return perm.list(),perm.signature()
exprs1=[]
for i in range(3):
    for j in range(3):
        upper_ind,upper_sig=gen_perm(i)
        lower_ind,lower_sig=gen_perm(j)
        sign=upper_sig*lower_sig
        ## fix position
        if i==2 and j==0:
            # swap ind
            assert upper_ind==[0,1,2]
            upper_ind=[1,0,2]
            sign*=-1
        if j==2 and i==0:
            # swap ind
            assert lower_ind==[0,1,2]
            lower_ind=[1,0,2]
            sign*=-1
        ## cumulant sign
        sign*=-1
        expr=sign*Tensor(gamma,(upper[upper_ind[-1]],),(lower[lower_ind[-1]],))*Tensor(
            gamma,(upper[upper_ind[0]],upper[upper_ind[1]]),(lower[lower_ind[0]],lower[lower_ind[1]])
        )
        exprs1.append(expr)

In [166]:
exprs1

[-\gamma((p,),(s,))*\gamma((q, r),(t, u)),
 \gamma((p,),(t,))*\gamma((q, r),(s, u)),
 \gamma((p,),(u,))*\gamma((q, r),(t, s)),
 \gamma((q,),(s,))*\gamma((p, r),(t, u)),
 -\gamma((q,),(t,))*\gamma((p, r),(s, u)),
 \gamma((q,),(u,))*\gamma((p, r),(s, t)),
 \gamma((r,),(s,))*\gamma((q, p),(t, u)),
 \gamma((r,),(t,))*\gamma((p, q),(s, u)),
 -\gamma((r,),(u,))*\gamma((p, q),(s, t))]

In [167]:
from sympy.combinatorics.named_groups import SymmetricGroup
upper=[p,q,r]
lower=[s,t,u]

In [168]:
s3=SymmetricGroup(3)    

In [169]:
exprs2=[]
for per in s3.generate():
    new_lower=[lower[per(i)] for i in range(3)]
    expr=[]
    for i in range(3):
        expr.append(Tensor(gamma,(upper[i],),(new_lower[i],)))  
    expr=2*per.signature()*Mul(*expr)
    exprs2.append(expr)

In [170]:
with open("cumulant_decomposition.tex","w") as f:
    f.write(latex(cum3)+"="+latex(dm3))
    f.write(latex(Add(*exprs1)))
    f.write("\\\\\n")
    f.write("+")
    f.write(latex(Add(*exprs2)))

## Now add cumulant term to final expression

In [221]:
final_expr=expr_sub_final+cum3-dm3-Add(*exprs1)-Add(*exprs2)

In [222]:
total_term=len(final_expr.args)

In [223]:
three_ind=[]
op2_type=[]
op1_type=[]
none_op_type=[]

In [224]:
for arg in final_expr.args:
    if isinstance(arg,Mul):
        op2_type_flag=False
        op1_type_flag=False
        for factor in arg.args:
            if isinstance(factor,Tensor) and factor.symbol==op and len(factor.upper)==2:
                op2_type_flag=True
            if isinstance(factor,Tensor) and factor.symbol==op and len(factor.upper)==1:
                op1_type_flag=True
        if op2_type_flag :
            op2_type.append(arg)
        elif op1_type_flag:
            op1_type.append(arg)
        else:
            none_op_type.append(arg)
    else:
        three_ind.append(arg)

In [312]:
three_ind

[\lambda((p, q, r),(s, t, u)), \tilde{a}((p, q, r),(s, t, u))]

In [313]:
op2_type

[\gamma((p,),(s,))*a((q, r),(t, u)),
 \gamma((q,),(t,))*a((p, r),(s, u)),
 \gamma((r,),(u,))*a((p, q),(s, t)),
 -\gamma((p,),(t,))*a((q, r),(s, u)),
 -\gamma((p,),(u,))*a((q, r),(t, s)),
 -\gamma((q,),(s,))*a((p, r),(t, u)),
 -\gamma((q,),(u,))*a((p, r),(s, t)),
 -\gamma((r,),(s,))*a((q, p),(t, u)),
 -\gamma((r,),(t,))*a((p, q),(s, u))]

In [315]:
op1_type

[\gamma((p, q),(s, t))*a((r,),(u,)),
 \gamma((p, r),(s, u))*a((q,),(t,)),
 \gamma((q, r),(t, u))*a((p,),(s,)),
 -\gamma((p, q),(s, u))*a((r,),(t,)),
 -\gamma((p, q),(u, t))*a((r,),(s,)),
 -\gamma((p, r),(s, t))*a((q,),(u,)),
 -\gamma((p, r),(t, u))*a((q,),(s,)),
 -\gamma((q, r),(s, u))*a((p,),(t,)),
 -\gamma((r, q),(s, t))*a((p,),(u,)),
 -2*\gamma((p,),(s,))*\gamma((q,),(t,))*a((r,),(u,)),
 -2*\gamma((p,),(s,))*\gamma((r,),(u,))*a((q,),(t,)),
 -2*\gamma((p,),(t,))*\gamma((q,),(u,))*a((r,),(s,)),
 -2*\gamma((p,),(t,))*\gamma((r,),(s,))*a((q,),(u,)),
 -2*\gamma((p,),(u,))*\gamma((q,),(s,))*a((r,),(t,)),
 -2*\gamma((p,),(u,))*\gamma((r,),(t,))*a((q,),(s,)),
 -2*\gamma((q,),(s,))*\gamma((r,),(t,))*a((p,),(u,)),
 -2*\gamma((q,),(t,))*\gamma((r,),(u,))*a((p,),(s,)),
 -2*\gamma((q,),(u,))*\gamma((r,),(s,))*a((p,),(t,)),
 2*\gamma((p,),(s,))*\gamma((q,),(u,))*a((r,),(t,)),
 2*\gamma((p,),(s,))*\gamma((r,),(t,))*a((q,),(u,)),
 2*\gamma((p,),(t,))*\gamma((q,),(s,))*a((r,),(u,)),
 2*\gamma((p,),(

In [231]:
none_op_type

[\gamma((p,),(t,))*\gamma((q, r),(s, u)),
 \gamma((p,),(u,))*\gamma((r, q),(s, t)),
 \gamma((q,),(s,))*\gamma((p, r),(t, u)),
 \gamma((q,),(u,))*\gamma((p, r),(s, t)),
 \gamma((r,),(s,))*\gamma((p, q),(u, t)),
 \gamma((r,),(t,))*\gamma((p, q),(s, u)),
 -\gamma((p,),(s,))*\gamma((q, r),(t, u)),
 -\gamma((q,),(t,))*\gamma((p, r),(s, u)),
 -\gamma((r,),(u,))*\gamma((p, q),(s, t)),
 -4*\gamma((p,),(s,))*\gamma((q,),(u,))*\gamma((r,),(t,)),
 -4*\gamma((p,),(t,))*\gamma((q,),(s,))*\gamma((r,),(u,)),
 -4*\gamma((p,),(u,))*\gamma((q,),(t,))*\gamma((r,),(s,)),
 4*\gamma((p,),(s,))*\gamma((q,),(t,))*\gamma((r,),(u,)),
 4*\gamma((p,),(t,))*\gamma((q,),(u,))*\gamma((r,),(s,)),
 4*\gamma((p,),(u,))*\gamma((q,),(s,))*\gamma((r,),(t,))]

In [232]:
assert len(none_op_type)+len(op1_type)+len(op2_type)+len(three_ind)==total_term

In [235]:
with open("operator_decomposition.tex","w") as f:
    f.write(latex(op3)+"=")
    f.write(latex(three_ind[1])+"+")
    f.write(latex(Add(*op2_type)))
    f.write("\\\\\n")
    f.write("+")
    f.write(latex(Add(*op1_type[:9])))
    f.write("\\\\\n")
    f.write(latex(Add(*op1_type[9:18])))
    f.write("\\\\\n")
    f.write(latex(Add(*op1_type[18:])))
    f.write("\\\\\n")
    f.write(latex(Add(*none_op_type[:9])))
    f.write("\\\\\n")
    f.write("+"+latex(Add(*none_op_type[9:])))
    f.write("+"+latex(three_ind[0]))

## Do spin adaptation

#### Spin adaptation can be done by classification of index pairs that are  keeped pair with original term

In [430]:
sf_dm,sf_op=symbols("D E")
sf_operator=Tensor(sf_op,upper,lower)

In [379]:
op3

a((p, q, r),(s, t, u))

In [268]:
all_pair_term=op2_type[0]

In [269]:
summations=[]

In [290]:
pa,pb,qa,qb,ra,rb,sa,sb,ta,tb,ua,ub=symbols("p_a p_b q_a q_b r_a r_b s_a s_b t_a t_b u_a u_b")
p_sind=[pa,pb]
q_sind=[qa,qb]
r_sind=[ra,rb]
s_sind=[sa,sb]
t_sind=[ta,tb]
u_sind=[ua,ub]

## all pairs are not changed

In [306]:
spin_terms=[]
for i in range(2):
    for j in range(2):
        for k in range(2):
            spin_term=all_pair_term.subs(p,p_sind[i]).subs(q,q_sind[j]).subs(r,r_sind[k]).subs(s,s_sind[i]).subs(t,t_sind[j]).subs(u,u_sind[k])
            spin_terms.append(spin_term)

In [307]:
Add(*spin_terms)

\gamma((p_a,),(s_a,))*a((q_a, r_a),(t_a, u_a)) + \gamma((p_a,),(s_a,))*a((q_a, r_b),(t_a, u_b)) + \gamma((p_a,),(s_a,))*a((q_b, r_a),(t_b, u_a)) + \gamma((p_a,),(s_a,))*a((q_b, r_b),(t_b, u_b)) + \gamma((p_b,),(s_b,))*a((q_a, r_a),(t_a, u_a)) + \gamma((p_b,),(s_b,))*a((q_a, r_b),(t_a, u_b)) + \gamma((p_b,),(s_b,))*a((q_b, r_a),(t_b, u_a)) + \gamma((p_b,),(s_b,))*a((q_b, r_b),(t_b, u_b))

## one pair is not changed

In [308]:
one_pair_term=op2_type[4]
spin_terms=[]
for i in range(2):
    for j in range(2):
        for k in range(2):
            spin_term=one_pair_term.subs(p,p_sind[i]).subs(q,q_sind[j]).subs(r,r_sind[k]).subs(s,s_sind[i]).subs(t,t_sind[j]).subs(u,u_sind[k])
            spin_terms.append(spin_term)

In [309]:
Add(*spin_terms)

-\gamma((p_a,),(u_a,))*a((q_a, r_a),(t_a, s_a)) - \gamma((p_a,),(u_a,))*a((q_b, r_a),(t_b, s_a)) - \gamma((p_a,),(u_b,))*a((q_a, r_b),(t_a, s_a)) - \gamma((p_a,),(u_b,))*a((q_b, r_b),(t_b, s_a)) - \gamma((p_b,),(u_a,))*a((q_a, r_a),(t_a, s_b)) - \gamma((p_b,),(u_a,))*a((q_b, r_a),(t_b, s_b)) - \gamma((p_b,),(u_b,))*a((q_a, r_b),(t_a, s_b)) - \gamma((p_b,),(u_b,))*a((q_b, r_b),(t_b, s_b))

## all pairs are change

In [310]:
one_pair_term=none_op_type[-1]
spin_terms=[]
for i in range(2):
    for j in range(2):
        for k in range(2):
            spin_term=one_pair_term.subs(p,p_sind[i]).subs(q,q_sind[j]).subs(r,r_sind[k]).subs(s,s_sind[i]).subs(t,t_sind[j]).subs(u,u_sind[k])
            spin_terms.append(spin_term)

In [311]:
Add(*spin_terms)

4*\gamma((p_a,),(u_a,))*\gamma((q_a,),(s_a,))*\gamma((r_a,),(t_a,)) + 4*\gamma((p_a,),(u_a,))*\gamma((q_b,),(s_a,))*\gamma((r_a,),(t_b,)) + 4*\gamma((p_a,),(u_b,))*\gamma((q_a,),(s_a,))*\gamma((r_b,),(t_a,)) + 4*\gamma((p_a,),(u_b,))*\gamma((q_b,),(s_a,))*\gamma((r_b,),(t_b,)) + 4*\gamma((p_b,),(u_a,))*\gamma((q_a,),(s_b,))*\gamma((r_a,),(t_a,)) + 4*\gamma((p_b,),(u_a,))*\gamma((q_b,),(s_b,))*\gamma((r_a,),(t_b,)) + 4*\gamma((p_b,),(u_b,))*\gamma((q_a,),(s_b,))*\gamma((r_b,),(t_a,)) + 4*\gamma((p_b,),(u_b,))*\gamma((q_b,),(s_b,))*\gamma((r_b,),(t_b,))

In [316]:
## spin adapt op2 type

In [322]:
term=op2_type[0]

In [392]:
def determine_num_exchange_ind(term):
    upper_ind=[]
    lower_ind=[]
    for factor in term.args:
        if isinstance(factor,Tensor) :
            upper_ind.extend(factor.upper)
            lower_ind.extend(factor.lower)
    upper_position_encode=[]
    for ind in upper_ind:
        if ind==p:
            upper_position_encode.append(0)
        elif ind==q:
            upper_position_encode.append(1)
        elif ind==r:
            upper_position_encode.append(2)
        else:
            raise ValueError("unexpected ind")
    lower_position_encode=[]
    for ind in lower_ind:
        if ind==s:
            lower_position_encode.append(0)
        elif ind==t:
            lower_position_encode.append(1)
        elif ind==u:
            lower_position_encode.append(2)
        else:
            raise ValueError("unexpected ind")
    exchange_num=0
    for i in range(3):
        if upper_position_encode[i]==lower_position_encode[i]:
            exchange_num+=1
    return exchange_num

In [393]:
new_factor=[]
for factor in op2_type[0].args:
    if factor.symbol==op:
        sf_factor=factor.subs(factor.symbol,sf_op)
        new_factor.append(sf_factor)
    elif factor.symbol==gamma:
        sf_factor=factor.subs(factor.symbol,sf_dm)
        new_factor.append(sf_factor)
    else:
        new_factor.append(factor)

In [420]:
from sympy  import Rational,Integer
def spin_adapted_by_num(term):
    num=determine_num_exchange_ind(term)
    if num==3:
        spin_factor=Integer(1)
    if num==1:
        spin_factor=Rational(1,2)
    if num==0:
        spin_factor=Rational(1,4)
    
    if isinstance(term,Mul):
        new_factor=[]
        for factor in term.args:
            if isinstance(factor,Tensor):
                if factor.symbol==op:
                    sf_factor=factor.subs(factor.symbol,sf_op)
                    new_factor.append(sf_factor)
                elif factor.symbol==gamma:
                    sf_factor=factor.subs(factor.symbol,sf_dm)
                    new_factor.append(sf_factor)
            else:
                new_factor.append(factor)
        return Mul(*new_factor)*spin_factor
    else:
        raise ValueError("unexpected term type")

In [421]:
spin_free_term=[]
for term in op2_type:
    new_term=spin_adapted_by_num(term)
    spin_free_term.append(new_term)

In [422]:
sf_op2=Add(*spin_free_term)

In [423]:
spin_free_term=[]
for term in none_op_type:
    new_term=spin_adapted_by_num(term)
    spin_free_term.append(new_term)

In [424]:
sf_none_op=Add(*spin_free_term)

In [427]:
spin_terms_chunk=[op2_type,op1_type[:9],op1_type[9:18],
                op1_type[18:], none_op_type[:9],none_op_type[9:]]
spin_free_terms_chuck=[]
for chunk in spin_terms_chunk:
    spin_free_term=[]
    for term in chunk:
        new_term=spin_adapted_by_num(term)
        spin_free_term.append(new_term)
    spin_free_terms_chuck.append(Add(*spin_free_term))
    

In [428]:
spin_free_terms_chuck

[D((p,),(s,))*E((q, r),(t, u)) - D((p,),(t,))*E((q, r),(s, u))/2 - D((p,),(u,))*E((q, r),(t, s))/2 - D((q,),(s,))*E((p, r),(t, u))/2 + D((q,),(t,))*E((p, r),(s, u)) - D((q,),(u,))*E((p, r),(s, t))/2 - D((r,),(s,))*E((q, p),(t, u))/2 - D((r,),(t,))*E((p, q),(s, u))/2 + D((r,),(u,))*E((p, q),(s, t)),
 D((p, q),(s, t))*E((r,),(u,)) - D((p, q),(s, u))*E((r,),(t,))/2 - D((p, q),(u, t))*E((r,),(s,))/2 - D((p, r),(s, t))*E((q,),(u,))/2 + D((p, r),(s, u))*E((q,),(t,)) - D((p, r),(t, u))*E((q,),(s,))/2 - D((q, r),(s, u))*E((p,),(t,))/2 + D((q, r),(t, u))*E((p,),(s,)) - D((r, q),(s, t))*E((p,),(u,))/2,
 -2*D((p,),(s,))*D((q,),(t,))*E((r,),(u,)) - 2*D((p,),(s,))*D((r,),(u,))*E((q,),(t,)) - D((p,),(t,))*D((q,),(u,))*E((r,),(s,))/2 - D((p,),(t,))*D((r,),(s,))*E((q,),(u,))/2 - D((p,),(u,))*D((q,),(s,))*E((r,),(t,))/2 - D((p,),(u,))*D((r,),(t,))*E((q,),(s,))/2 - D((q,),(s,))*D((r,),(t,))*E((p,),(u,))/2 - 2*D((q,),(t,))*D((r,),(u,))*E((p,),(s,)) - D((q,),(u,))*D((r,),(s,))*E((p,),(t,))/2,
 D((p,),(s,)

In [434]:
with open("spin_free.tex","w") as f:
    f.write(latex(sf_operator)+"\\approx \n")
    for ind,chunk in enumerate(spin_free_terms_chuck):
        if ind%2==1:
            f.write("+")
        f.write(latex(chunk))
        f.write("\\\\\n")